In [1]:
from llama_cpp import Llama

In [3]:
import os
# Get the number of physical cores on your machine
cores = os.cpu_count()

# For example, if you have 8 cores, 4 or 6 is usually the sweet spot.
cores_to_use = cores // 2 
print(f"Cores on this machine: {cores}")
print(f"Cores to use: {cores_to_use}")

Cores on this machine: 2
Cores to use: 1


### **Without LoRA**

In [4]:
llm = Llama(
    model_path='gguf_models/phi4_mini_instruct_q4_k_m.gguf',
    n_ctx=2000,
    n_threads=cores_to_use,   # Increase this for faster generation!
    n_batch=512,       # Helps the CPU process the initial prompt faster
    n_gpu_layers=0,     # Explicitly disables GPU offloading
    use_mmap=True,     # ADD THIS: Critically important for low-RAM systems
    use_mlock=False    # Ensure this is False so it doesn't force-lock RAM
)

llama_model_loader: loaded meta data with 36 key-value pairs and 196 tensors from gguf_models/phi4_mini_instruct_q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:              phi3.rope.scaling.attn_factor f32              = 1.190238
llama_model_loader: - kv   2:                               general.type str              = model
llama_model_loader: - kv   3:                               general.name str              = Phi 4 Mini Instruct
llama_model_loader: - kv   4:                           general.finetune str              = instruct
llama_model_loader: - kv   5:                           general.basename str              = Phi-4
llama_model_loader: - kv   6:                         general.size_label str              = mini
llama_model_loader: - kv   7:                 

llama_model_loader: - kv  24:                      tokenizer.ggml.tokens arr[str,200064]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  25:                  tokenizer.ggml.token_type arr[i32,200064]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  26:                      tokenizer.ggml.merges arr[str,199742]  = ["Ġ Ġ", "ĠĠ ĠĠ", "i n", "e r", ...
llama_model_loader: - kv  27:                tokenizer.ggml.bos_token_id u32              = 199999
llama_model_loader: - kv  28:                tokenizer.ggml.eos_token_id u32              = 199999
llama_model_loader: - kv  29:            tokenizer.ggml.unknown_token_id u32              = 199999
llama_model_loader: - kv  30:            tokenizer.ggml.padding_token_id u32              = 199999
llama_model_loader: - kv  31:               tokenizer.ggml.add_bos_token bool             = false
llama_model_loader: - kv  32:               tokenizer.ggml.add_eos_token bool             = false
llama_model_loa

In [5]:
output = llm.create_chat_completion(
    messages =[
        {
            "role": "system", "content":"You are a helpful AI assistant specialised in African history which gives concise answers to questions asked."
        },
        {
            "role": "user", "content":"Briefly detail the significance story of the Igbo god Amadioha?"
        }
    ],
    max_tokens=128,
    temperature=0.1
)

llama_perf_context_print:        load time =  118839.06 ms
llama_perf_context_print: prompt eval time =  118838.76 ms /    38 tokens ( 3127.34 ms per token,     0.32 tokens per second)
llama_perf_context_print:        eval time =  309037.78 ms /    63 runs   ( 4905.36 ms per token,     0.20 tokens per second)
llama_perf_context_print:       total time =  428135.38 ms /   101 tokens
llama_perf_context_print:    graphs reused =         60


In [6]:
output

{'id': 'chatcmpl-e4e81de4-0c63-47ef-8715-d4325d2170be',
 'object': 'chat.completion',
 'created': 1774116153,
 'model': 'gguf_models/phi4_mini_instruct_q4_k_m.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "Amadioha is a lesser-known deity in Igbo mythology, often associated with thunder and lightning. He is believed to be the god of justice and thunder, and his presence is invoked during thunderstorms to ensure justice and balance. Amadioha's story highlights the importance of maintaining harmony and fairness in Igbo society."},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 38, 'completion_tokens': 63, 'total_tokens': 101}}

In [7]:
print(output["choices"][0]["message"]["content"])

Amadioha is a lesser-known deity in Igbo mythology, often associated with thunder and lightning. He is believed to be the god of justice and thunder, and his presence is invoked during thunderstorms to ensure justice and balance. Amadioha's story highlights the importance of maintaining harmony and fairness in Igbo society.


In [8]:
# using streaming output

output_stream = llm.create_chat_completion(
    messages =[
        {
            "role": "system", "content":"You are a helpful AI assistant specialised in African history which gives concise answers to questions asked."
        },
        {
            "role": "user", "content":"Briefly detail the significance story of the Igbo god Amadioha?"
        }
    ],
    max_tokens=128,
    temperature=0.1,
    stream=True
)


for chunk in output_stream:
    delta = chunk["choices"][0]["delta"]
    if "content" in delta:
        print(delta["content"], end="", flush=True)
print(output_stream)

Llama.generate: 37 prefix-match hit, remaining 1 prompt tokens to eval


Amadioha is a lesser-known deity in Igbo mythology, often associated with thunder and lightning. He is considered the god of justice and is believed to punish wrongdoers. Amadioha's story highlights the importance of justice and moral integrity in Igbo culture.

llama_perf_context_print:        load time =  118839.06 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  274469.36 ms /    55 runs   ( 4990.35 ms per token,     0.20 tokens per second)
llama_perf_context_print:       total time =  274712.20 ms /    56 tokens
llama_perf_context_print:    graphs reused =         53


<generator object _convert_text_completion_chunks_to_chat at 0x000002B2E170EF80>


### **With LoRA**

In [9]:
llm_with_lora = Llama(
    model_path='gguf_models/phi4_mini_instruct_q4_k_m.gguf',
    lora_path='gguf_models/lorafrica_lora_adapter.gguf',
    n_ctx=2000,
    n_threads=cores_to_use,   # Increase this for faster generation!
    n_batch=512,       # Helps the CPU process the initial prompt faster
    n_gpu_layers=0,     # Explicitly disables GPU offloading
    use_mmap=True,     # ADD THIS: Critically important for low-RAM systems
    use_mlock=False    # Ensure this is False so it doesn't force-lock RAM
)

llama_model_loader: loaded meta data with 36 key-value pairs and 196 tensors from gguf_models/phi4_mini_instruct_q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:              phi3.rope.scaling.attn_factor f32              = 1.190238
llama_model_loader: - kv   2:                               general.type str              = model
llama_model_loader: - kv   3:                               general.name str              = Phi 4 Mini Instruct
llama_model_loader: - kv   4:                           general.finetune str              = instruct
llama_model_loader: - kv   5:                           general.basename str              = Phi-4
llama_model_loader: - kv   6:                         general.size_label str              = mini
llama_model_loader: - kv   7:                 

In [10]:
output = llm_with_lora.create_chat_completion(
    messages =[
        {
            "role": "system", "content":"You are a helpful AI assistant specialised in African history which gives concise answers to questions asked."
        },
        {
            "role": "user", "content":"Briefly detail the significance story of the Igbo god Amadioha?"
        }
    ],
    max_tokens=128,
    temperature=0.1
)

llama_perf_context_print:        load time =  111002.65 ms
llama_perf_context_print: prompt eval time =  110997.34 ms /    38 tokens ( 2920.98 ms per token,     0.34 tokens per second)
llama_perf_context_print:        eval time =  416919.07 ms /    84 runs   ( 4963.32 ms per token,     0.20 tokens per second)
llama_perf_context_print:       total time =  528251.35 ms /   122 tokens
llama_perf_context_print:    graphs reused =         81


In [11]:
output

{'id': 'chatcmpl-c4be0a3c-9a11-4ad5-bc61-e43c9069e11b',
 'object': 'chat.completion',
 'created': 1774116878,
 'model': 'gguf_models/phi4_mini_instruct_q4_k_m.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "Amadioha is a significant deity in Igbo mythology, representing justice and the thunder god. He is believed to reside in the heavens and is often invoked to settle disputes and mete out justice. Amadioha's thunderbolts are said to strike down those who commit injustices, symbolizing his role as a divine enforcer of moral order. His story underscores the importance of justice and fairness in Igbo culture."},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 38, 'completion_tokens': 84, 'total_tokens': 122}}

In [12]:
print(output["choices"][0]["message"]["content"])

Amadioha is a significant deity in Igbo mythology, representing justice and the thunder god. He is believed to reside in the heavens and is often invoked to settle disputes and mete out justice. Amadioha's thunderbolts are said to strike down those who commit injustices, symbolizing his role as a divine enforcer of moral order. His story underscores the importance of justice and fairness in Igbo culture.


In [13]:
# using streaming output

output_stream = llm_with_lora.create_chat_completion(
    messages =[
        {
            "role": "system", "content":"You are a helpful AI assistant specialised in African history which gives concise answers to questions asked."
        },
        {
            "role": "user", "content":"Briefly detail the significance story of the Igbo god Amadioha?"
        }
    ],
    max_tokens=128,
    temperature=0.1,
    stream=True
)


for chunk in output_stream:
    delta = chunk["choices"][0]["delta"]
    if "content" in delta:
        print(delta["content"], end="", flush=True)

Llama.generate: 37 prefix-match hit, remaining 1 prompt tokens to eval


Amadioha is a significant deity in Igbo mythology, representing justice and the thunder god. He is believed to reside in the heavens and is often invoked to settle disputes and mete out punishment for wrongdoing. Amadioha's thunderbolts are said to strike down those who commit injustices, symbolizing divine retribution. His presence underscores the importance of justice and moral order in Igbo culture.

llama_perf_context_print:        load time =  111002.65 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  643799.49 ms /    81 runs   ( 7948.14 ms per token,     0.13 tokens per second)
llama_perf_context_print:       total time =  644416.19 ms /    82 tokens
llama_perf_context_print:    graphs reused =         78
